# 🏢 Caso de Estudio: LegacyTech Solutions

## 🎯 Objetivo
Simular la aplicación del framework EBLET Enterprise en una empresa real (ficticia) para demostrar su utilidad práctica, desde el diagnóstico hasta el cálculo de ROI.

## 📋 La Empresa: LegacyTech Solutions
- **Sector:** Tecnología / Software
- **Tamaño:** 180 empleados
- **Cultura Declarada (Dirección):**  Adhocracia ("Somos ágiles e innovadores")
- **Cultura Real (Entorno):** 🟡 Jerárquica (Procesos rígidos, burocracia)
- **Perfil de Riesgo:** 🔵 Riesgo Boreout

## 🚀 Flujo del Análisis
1. Generación de la empresa y sus empleados
2. Cálculo de KPIs y clasificación vs Benchmark
3. Diagnóstico de Brecha Cultural
4. Análisis de Costes de Rotación
5. Informe Ejecutivo y Recomendaciones

In [2]:

# IMPORTS Y CONFIGURACIÓN


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

import os
import sys

RAIZ_PROYECTO = r"C:\Users\torre\OneDrive\Escritorio\EBLET-People-Analytics\Python"
sys.path.insert(0, os.path.join(RAIZ_PROYECTO, "src"))

# Paleta de colores
COLORES_ESCENARIOS = {
    "saludable": "#2ecc71", "estable": "#f1c40f",
    "riesgo_burnout": "#e67e22", "riesgo_boreout": "#3498db", "critico": "#e74c3c"
}

print("✅ Notebook 3 cargado correctamente")

✅ Notebook 3 cargado correctamente


## 1.  Generación de LegacyTech Solutions

Generamos 180 empleados con un perfil de **Riesgo Boreout** en una cultura **Jerárquica**.

In [3]:

# GENERACIÓN DE LA EMPRESA (180 EMPLEADOS)


from encuesta import generar_respuestas_encuesta

n_empleados = 180
np.random.seed(42)

print("🔧 Generando LegacyTech usando el motor de EBLET...")

# 1. Crear DataFrame base con metadatos y estados latentes
# 🆕 Valores moderados para perfil "Riesgo Boreout" (no extremo)
df_base = pd.DataFrame({
    "empleado_id": range(1, n_empleados + 1),
    "empresa": "LegacyTech Solutions",
    "cultura": "Jerarquica",  # Cultura real del entorno
    # Estados latentes MODERADOS
    "L_burnout": 2.2,    # Burnout bajo-medio (antes 1.8)
    "L_boreout": 3.8,    # Boreout alto-medio (antes 4.2)
    "L_wellbeing": 2.6,  # Bienestar bajo-medio (antes 2.3)
    "L_rotation": 3.6    # Rotación media-alta (antes 4.0)
})

# 🆕 Aumentar variabilidad individual (std de 0.3 a 0.5)
df_base["L_burnout"] = df_base["L_burnout"] + np.random.normal(0, 0.5, n_empleados)
df_base["L_boreout"] = df_base["L_boreout"] + np.random.normal(0, 0.5, n_empleados)
df_base["L_wellbeing"] = df_base["L_wellbeing"] + np.random.normal(0, 0.5, n_empleados)
df_base["L_rotation"] = df_base["L_rotation"] + np.random.normal(0, 0.5, n_empleados)

# Clip para mantener valores en rango razonable
for col in ["L_burnout", "L_boreout", "L_wellbeing", "L_rotation"]:
    df_base[col] = df_base[col].clip(1, 5)

# 2. Generar las 72 preguntas usando la función real
df_respuestas = generar_respuestas_encuesta(df_base)

# 3. Combinar metadatos + respuestas
df_legacy = pd.concat([df_base, df_respuestas], axis=1)

# 4. Añadir cultura declarada (para análisis de brecha)
df_legacy["cultura_declarada"] = "Adhocracia"  # Lo que la dirección cree

# 5. Simular salarios para cálculo de costes
df_legacy["salario_anual"] = np.random.normal(35000, 10000, n_empleados).clip(20000, 60000).round(0)

print(f"✅ Empresa generada: {len(df_legacy)} empleados")
print(f"   Cultura Real: {df_legacy['cultura'].unique()[0]}")
print(f"   Cultura Declarada: {df_legacy['cultura_declarada'].unique()[0]}")
print(f"   Columnas totales: {len(df_legacy.columns)}")
print(f"   Preguntas generadas: {len([c for c in df_legacy.columns if c.startswith('q')])}")

🔧 Generando LegacyTech usando el motor de EBLET...
✅ Empresa generada: 180 empleados
   Cultura Real: Jerarquica
   Cultura Declarada: Adhocracia
   Columnas totales: 81
   Preguntas generadas: 72


## 2. 📊 Cálculo de KPIs y Clasificación

Calculamos los KPIs directamente desde las preguntas para garantizar la precisión.

In [ ]:

# CÁLCULO DE KPIs DESDE CERO
# 

print("🔧 Calculando KPIs desde las preguntas de la encuesta...")

df_legacy["kpi_burnout"] = df_legacy[["q16", "q19", "q23", "q26", "q28"]].mean(axis=1)
df_legacy["kpi_boreout"] = df_legacy[["q37", "q39", "q41", "q43"]].mean(axis=1)
df_legacy["kpi_bienestar"] = df_legacy[["q45", "q46", "q47", "q48"]].mean(axis=1)
df_legacy["kpi_rotacion"] = df_legacy[["q57", "q58", "q59"]].mean(axis=1)
df_legacy["kpi_contexto"] = df_legacy["q2"]

# Calcular cultura percibida
df_legacy["cvf_adhocracia"] = df_legacy[["q65", "q66"]].mean(axis=1)
df_legacy["cvf_clan"] = df_legacy[["q67", "q68"]].mean(axis=1)
df_legacy["cvf_mercado"] = df_legacy[["q69", "q70"]].mean(axis=1)
df_legacy["cvf_jerarquica"] = df_legacy[["q71", "q72"]].mean(axis=1)

# Resumen de KPIs de la empresa
kpi_cols = ["kpi_burnout", "kpi_boreout", "kpi_bienestar", "kpi_rotacion", "kpi_contexto"]
medias_legacy = df_legacy[kpi_cols].mean()

print("="*70)
print("📊 KPIs MEDIOS DE LEGACYTECH")
print("="*70)
for kpi, valor in medias_legacy.items():
    print(f"   {kpi:15s}: {valor:.2f}")

# Clasificación de empleados en perfiles individuales
from clasificador_individual import clasificar_individuo

perfiles = []
for _, row in df_legacy.iterrows():
    kpis_ind = {
        "burnout": row["kpi_burnout"], "boreout": row["kpi_boreout"],
        "bienestar": row["kpi_bienestar"], "rotacion": row["kpi_rotacion"],
        "cultura_dominante": row[["cvf_adhocracia", "cvf_clan", "cvf_mercado", "cvf_jerarquica"]].idxmax().replace("cvf_", "")
    }
    perfil = clasificar_individuo(kpis_ind)
    perfiles.append(perfil["nombre"])

df_legacy["perfil_individual"] = perfiles

print("\n👥 Distribución de perfiles individuales:")
print(df_legacy["perfil_individual"].value_counts().to_string())

🔧 Calculando KPIs desde las preguntas de la encuesta...
📊 KPIs MEDIOS DE LEGACYTECH
   kpi_burnout    : 2.08
   kpi_boreout    : 3.82
   kpi_bienestar  : 2.57
   kpi_rotacion   : 3.58
   kpi_contexto   : 2.40

👥 Distribución de perfiles individuales:
perfil_individual
🔵 Aburrido Crónico    115
🟡 Estable Neutro       42
⚫ Desvinculado         23


## 3. 🏛️ Diagnóstico de Brecha Cultural

Aquí reside el **valor diferencial** de EBLET: medir la diferencia entre la cultura que la empresa *cree* que tiene y la que los empleados *realmente perciben*.

In [5]:

# BRECHA CULTURAL (DECLARADA vs PERCIBIDA)


cultura_declarada = df_legacy["cultura_declarada"].unique()[0] # Adhocracia

# Scores percibidos
scores_percibidos = {
    "Adhocracia": df_legacy["cvf_adhocracia"].mean(),
    "Clan": df_legacy["cvf_clan"].mean(),
    "Mercado": df_legacy["cvf_mercado"].mean(),
    "Jerarquica": df_legacy["cvf_jerarquica"].mean()
}

cultura_percibida = max(scores_percibidos, key=scores_percibidos.get)
score_percibido_max = scores_percibidos[cultura_percibida]
score_declarada = scores_percibidos[cultura_declarada]

# Índice de coherencia
if cultura_declarada == cultura_percibida:
    indice_coherencia = 100
else:
    indice_coherencia = (score_declarada / score_percibido_max) * 100

print("="*70)
print("🏛️ DIAGNÓSTICO DE BRECHA CULTURAL")
print("="*70)
print(f"📋 Cultura DECLARADA (Dirección): {cultura_declarada} (Score percibido: {score_declarada:.2f})")
print(f"👥 Cultura PERCIBIDA (Empleados): {cultura_percibida} (Score percibido: {score_percibido_max:.2f})")
print(f"️  Índice de Coherencia: {indice_coherencia:.1f}%")

# Radar Chart
culturas_lista = ["Adhocracia", "Clan", "Mercado", "Jerarquica", "Adhocracia"]
scores_p = [scores_percibidos[c] for c in culturas_lista[:-1]] + [scores_percibidos["Adhocracia"]]

# Simular scores declarados (la empresa se da un 4.5 en su cultura ideal)
scores_d = [4.5 if c == cultura_declarada else 2.0 for c in culturas_lista[:-1]] + [4.5]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(r=scores_p, theta=culturas_lista, fill='toself', name='👥 Percibida (Realidad)', line_color='#e74c3c', opacity=0.6))
fig.add_trace(go.Scatterpolar(r=scores_d, theta=culturas_lista, fill='toself', name='📋 Declarada (Ideal)', line_color='#3498db', opacity=0.3))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 5])),
    title=f"<b>🏛️ Brecha Cultural en LegacyTech</b><br>Coherencia: {indice_coherencia:.0f}%",
    height=550, width=750
)
fig.show()

print(f"\n💡 CONCLUSIÓN: La empresa se ve como {cultura_declarada}, pero los empleados sienten {cultura_percibida}. Esta brecha es la causa raíz del Boreout.")

🏛️ DIAGNÓSTICO DE BRECHA CULTURAL
📋 Cultura DECLARADA (Dirección): Adhocracia (Score percibido: 1.98)
👥 Cultura PERCIBIDA (Empleados): Jerarquica (Score percibido: 4.01)
️  Índice de Coherencia: 49.3%



💡 CONCLUSIÓN: La empresa se ve como Adhocracia, pero los empleados sienten Jerarquica. Esta brecha es la causa raíz del Boreout.


## 4. 💰 Análisis de Costes de Rotación

¿Cuánto le cuesta a LegacyTech el problema detectado?

In [6]:

# COSTES DE ROTACIÓN


# 🆕 PASO 1: Primero crear la columna en df_legacy
df_legacy["coste_rotacion"] = df_legacy["salario_anual"] * 0.50

# 🆕 PASO 2: DESPUÉS filtrar (así df_bajas hereda la columna)
df_bajas = df_legacy[df_legacy["kpi_rotacion"] >= 3.5]
n_bajas_estimadas = len(df_bajas)

# PASO 3: Calcular costes
coste_total = df_bajas["coste_rotacion"].sum()
coste_medio = df_bajas["coste_rotacion"].mean() if len(df_bajas) > 0 else 0

# ROI de intervención (si reducimos la rotación un 30%)
ahorro_potencial = coste_total * 0.30

print("="*70)
print("💰 ANÁLISIS DE COSTES DE ROTACIÓN")
print("="*70)
print(f"👥 Empleados en riesgo de rotación (Intención > 3.5): {n_bajas_estimadas} ({n_bajas_estimadas/len(df_legacy)*100:.1f}%)")
print(f"💵 Coste medio por baja estimada: {coste_medio:,.0f}€")
print(f" Coste total estimado de rotación: {coste_total:,.0f}€")
print(f"🎯 Ahorro potencial (reducción del 30%): {ahorro_potencial:,.0f}€")

# Gráfico de barras
fig = px.bar(
    x=["Coste Actual de Rotación", "Ahorro Potencial (30%)"],
    y=[coste_total, ahorro_potencial],
    color=["Coste Actual", "Ahorro"],
    color_discrete_map={"Coste Actual": "#e74c3c", "Ahorro": "#2ecc71"},
    title="💰 Impacto Económico de la Rotación en LegacyTech",
    text_auto='€,.0f'
)
fig.update_layout(showlegend=False, height=450, width=700)
fig.show()

💰 ANÁLISIS DE COSTES DE ROTACIÓN
👥 Empleados en riesgo de rotación (Intención > 3.5): 98 (54.4%)
💵 Coste medio por baja estimada: 17,186€
 Coste total estimado de rotación: 1,684,196€
🎯 Ahorro potencial (reducción del 30%): 505,259€


## 5. 📄 Informe Ejecutivo Final

Resumen de hallazgos y recomendaciones para la dirección de LegacyTech.

In [7]:

# INFORME EJECUTIVO


print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    📄 INFORME EJECUTIVO - LEGACYTECH SOLUTIONS               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  1. DIAGNÓSTICO GENERAL:                                                     ║
║     • Perfil Organizacional:  RIESGO BOREOUT                               ║
║     • Burnout medio: 1.79 (Bajo) | Boreout medio: 4.20 (Muy Alto)            ║
║     • Bienestar medio: 2.32 (Bajo) | Rotación media: 3.98 (Alto)             ║
║                                                                              ║
║  2. HALLAZGO CLAVE - BRECHA CULTURAL:                                        ║
║     • Cultura Declarada: 🔵 Adhocracia (Innovación)                          ║
║     • Cultura Percibida: 🟡 Jerárquica (Rigidez)                             ║
║     • Índice de Coherencia: 54% (⚠️ CRÍTICO)                                 ║
║     • Causa Raíz: La burocracia frena la innovación prometida, generando     ║
║       frustración y aburrimiento crónico en los empleados.                   ║
║                                                                              ║
║  3. IMPACTO ECONÓMICO:                                                       ║
║     • Empleados en riesgo de fuga: """ + f"{n_bajas_estimadas}" + """                                               ║
║     • Coste total estimado de rotación: """ + f"{coste_total:,.0f}€" + """                                     ║
║     • Ahorro potencial con intervención (30%): """ + f"{ahorro_potencial:,.0f}€" + """                             ║
║                                                                              ║
║  4. RECOMENDACIONES ESTRATÉGICAS:                                            ║
║     1. AUDITORÍA DE PROCESOS: Eliminar burocracia innecesaria.               ║
║     2. JOB CRAFTING: Dar autonomía a los equipos para experimentar.          ║
║     3. LIDERAZGO: Formar a managers en cultura Adhocracia.                   ║
║     4. COMUNICACIÓN: Alinear el discurso directivo con la realidad operativa.║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")
print("✅ NOTEBOOK 03 COMPLETADO")


╔══════════════════════════════════════════════════════════════════════════════╗
║                    📄 INFORME EJECUTIVO - LEGACYTECH SOLUTIONS               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  1. DIAGNÓSTICO GENERAL:                                                     ║
║     • Perfil Organizacional:  RIESGO BOREOUT                               ║
║     • Burnout medio: 1.79 (Bajo) | Boreout medio: 4.20 (Muy Alto)            ║
║     • Bienestar medio: 2.32 (Bajo) | Rotación media: 3.98 (Alto)             ║
║                                                                              ║
║  2. HALLAZGO CLAVE - BRECHA CULTURAL:                                        ║
║     • Cultura Declarada: 🔵 Adhocracia (Innovación)                          ║
║     • Cultura Percibida: 🟡 Jerárquica (Rigidez)                             ║
║     • Índice de Coherencia: 54